### GPU Verification
Run this cell after setting your hardware accelerator to GPU to ensure it is connected.

In [ ]:
import torch

if torch.cuda.is_available():
    print(f" GPU connected: {torch.cuda.get_device_name(0)}")
else:
    print(" No GPU connected. Please go to Runtime > Change runtime type and select a GPU.")

# Preparing Text Data


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import sys

module_path = '/content/drive/MyDrive/LM'
if module_path not in sys.path:
    sys.path.append(module_path)

from datasets import load_dataset, Dataset, load_from_disk
from BPETokenizer import BPETokenizer as tk
from SlidingWindowDataset import create_data_loader
from TokenAndPositionEmbedding import TokenAndPositionEmbedding
import itertools
import torch

In [ ]:
SLICE_SIZE = 1_000_000
VOCAB_TRAIN_SIZE = 50_000
NUM_MERGES = 1000

CONTEXT_SIZE = 1024
STRIDE = 1024
EMBED_DIM = 768
BATCH_SIZE = 16

REGEX_PATTERN = r"""'(?:[sdmt]|ll|ve|re)| ?[a-zA-Z]+| ?[0-9]+| ?[^\sA-Za-z0-9]+|\s+(?!\S)|\s+"""

### Data Ingestion & Caching

In [ ]:


# Define a path on your Google Drive
gdrive_save_path = "/content/drive/MyDrive/LM/local_fineweb_data/hf_format"

if not os.path.exists(gdrive_save_path):
    print(f"Streaming {SLICE_SIZE:,} documents from FineWeb-Edu...")
    dataset_dict = load_dataset("HuggingFaceFW/fineweb-edu", name="sample-10BT", streaming=True)

    # Use a generator to avoid loading everything into RAM at once
    def data_generator():
        for example in dataset_dict['train'].take(SLICE_SIZE):
            yield example

    print("Converting dataset using memory-efficient generator...")
    local_dataset = Dataset.from_generator(data_generator)

    print(f"Saving dataset to Google Drive: {gdrive_save_path}")
    os.makedirs(os.path.dirname(gdrive_save_path), exist_ok=True)
    local_dataset.save_to_disk(gdrive_save_path)
else:
    print("Loading existing dataset from Google Drive...")

local_dataset = load_from_disk(gdrive_save_path)
print(f"Dataset ready. Total documents: {len(local_dataset):,}")

### Parallel Tokenization

In [ ]:
TOKENIZER_PATH = "bpe_tokenizer.json"

print("Checkinf for existing tokenizer")
tokenizer = tk.load(TOKENIZER_PATH)

if tokenizer is None:
  print(f"Training BPPE Tokenizer on a subset of {VOCAB_TRAIN_SIZE:, } documents")
  vocab_sub_sample =  local_dataset.select(range(VOCAB_TRAIN_SIZE))
  tokenizer = tk(REGEX_PATTERN, num_merges=NUM_MERGES)
  tokenizer.build_vocab(vocab_sub_sample)

  tokenizer.save(TOKENIZER_PATH)

else:
  print("Pre-trained loaded successfully.")

print(f"Vocabulary completed. Total vocabulary size: {tokenizer.vocab_size:,}")

print("Encoding dataset in parallel")
tokenized_dataset = local_dataset.map(tokenizer.encode_row, num_proc=os.cpu_count())

print("Flattening token IDs into a single master sequence")
all_token_ids = list(itertools.chain.from_iterable(tokenized_dataset['token_ids']))
print(f"flattening completed. Sequence length: {len(all_token_ids):,} tokens.")


### Sanity Check - Autogressive

In [ ]:
test_context_size = 4
enc_sample = all_token_ids[100:]
x = enc_sample[:test_context_size]
y = enc_sample[1:test_context_size + 1]

print("Visualizing Autoregressive Targets:")
for index in range(1, test_context_size + 1):
    context = enc_sample[:index]
    desired = enc_sample[index]
    print(f"{tokenizer.decode(context)}  --------->  {tokenizer.decode([desired])}"

### PyTorch DataLoader Initialization

In [ ]:
print(f"Creating DataLoader with Context Size: {CONTEXT_SIZE} and Stride: {STRIDE}...")
dataloader = create_data_loader(
    all_token_ids,
    batch_size=BATCH_SIZE,
    context_size=CONTEXT_SIZE,
    stride=STRIDE,
    shuffle=True,
    drop_last=True
)
print(f"DataLoader created successfully with {len(dataloader):,} batches.")

### Sanity Check - Batch Alignment

In [ ]:
x_batch, y_batch = next(iter(dataloader))

print("Input Batch Shape (X):", x_batch.shape)
print("Target Batch Shape (Y):", y_batch.shape)

# Verify that Y is shifted exactly 1 position to the right of X
print("\nFirst row, first 5 tokens of X:", x_batch[0, :5].tolist())
print("First row, first 5 tokens of Y:", y_batch[0, :5].tolist())

# Mathematically assert the shift for the entire sequence (up to the last token of X)
assert torch.all(x_batch[:, 1:] == y_batch[:, :-1]), "ERROR: Y is not properly shifted!"
print("\n✅ Assertion Passed: Y is correctly shifted by 1 token.")

### Positional Embeddings

In [ ]:
# Initialize the model's first structural layer
embedding_layer = TokenAndPositionEmbedding(
    vocab_size=tokenizer.vocab_size,
    embed_dim=EMBED_DIM,
    context_size=CONTEXT_SIZE
)


### Sanity Check - Embedding Shapes

In [ ]:

# Pass our test batch through the embedding layer
embedded_batch = embedding_layer(x_batch)

print("--- FINAL PIPELINE VERIFICATION ---")
print(f"Pre-Embedding Shape (X):   {x_batch.shape}")          # [BATCH_SIZE, CONTEXT_SIZE]
print(f"Post-Embedding Shape:      {embedded_batch.shape}")   # [BATCH_SIZE, CONTEXT_SIZE, EMBED_DIM]

# Verify dimensions match our GPT-2 specs
assert embedded_batch.shape == (BATCH_SIZE, CONTEXT_SIZE, EMBED_DIM), "ERROR: Dimensionality mismatch!"
print(f"✅ Target Dimensions Achieved: Batch={BATCH_SIZE}, Sequence={CONTEXT_SIZE}, Embed_Dim={EMBED_DIM}")
print("\nPipeline status: READY FOR MULTI-HEAD ATTENTION")